In [1]:
# resume_fft_detector.py
import random
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from ds import (
    WatermarkOnTheFlyDataset,
    discover_dataset_files,
    get_test_aug,
)
from model import make_model
from attack import (
    make_train_image_augmentations,
    make_clean_aug,
    make_jpeg_aug,
    make_msg_app_combo,
    make_down_up_attack,
    make_blur_aug,
    make_random_crop_attack,
    make_occlusion_block,
    make_geom_aug,
)
import time
import pandas as pd
from model import load_checkpoint
from watermark import get_watermarking_mask, get_watermarking_pattern
import diffusers
from diffusers import DPMSolverMultistepScheduler
from inverse_stable_diffusion import InversableStableDiffusionPipeline
from engine import eval_model_psnr_l1
from utils.thr import get_best_thrs
from sklearn import metrics
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore")

# ----------------- Config (adjust if needed) -----------------
NAME = "our_improved_meta"
DATA_DIR = "./verifier_dataset_stablediff_octoweb"
CHECKPOINT_PATH = "verifier_dataset_stablediff_octoweb_meta_verifier_300.pth"
EVAL_RESULT_SAVE_DIR = os.path.join("./eval_results/", NAME)
BATCH_SIZE = 8
NUM_WORKERS = 0
LR = 2e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
VALIDATION_SPLIT = 0.15
NUM_INFERENCE_STEPS = 50
GUIDANCE_SCALE = 7.5
SAVE_EVERY_EPOCHS = 1  # how often to save full checkpoint
IMAGE_SIZE = 512  # might adjust to your pipeline / VAE size
IMG_AUG = make_train_image_augmentations(IMAGE_SIZE)
TEST_AUG = get_test_aug(IMAGE_SIZE)
INCLUDE_MASK_PATCH = False
INCLUDE_PSNR_L1 = True  # whether to include PSNR metric in dataset output

# Watermarking parameters (should match those used during watermark embedding)
W_MASK_SHAPE = "circle"
W_CHANNEL = 0
W_RADIUS = 10
W_STRENGTH = 0.99
W_PATTERN = "octoweb"
# -------------------------------------------------------------

# ---------------- reproducibility ----------------
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
# -------------------------------------------------

# ---------------- Load or define PIPE and TEXT_EMBEDDINGS ----------------
# Validate that PIPE and TEXT_EMBEDDINGS are present (or load them here)

model_id = "stabilityai/stable-diffusion-2-1-base"
device = "cuda" if torch.cuda.is_available() else "cpu"
try:
    import torch
    import diffusers
    from diffusers import DPMSolverMultistepScheduler
    from inverse_stable_diffusion import InversableStableDiffusionPipeline

    model_id = "Manojb/stable-diffusion-2-1-base"
    device = "cuda" if torch.cuda.is_available() else "cpu"
    scheduler = DPMSolverMultistepScheduler.from_pretrained(
        model_id, subfolder="scheduler"
    )
    pipe = InversableStableDiffusionPipeline.from_pretrained(
        model_id,
        scheduler=scheduler,
        torch_dtype=torch.float16,
        # revision="fp16",
        verbose=False,
    )
    diffusers.utils.logging.disable_progress_bar()
    pipe.set_progress_bar_config(disable=True)
    pipe = pipe.to(device)

    TEXT_EMBEDDINGS = pipe.get_text_embedding("")  #
    PIPE = pipe  # make sure 'pipe' is in scope
except NameError:
    raise RuntimeError(
        "Please ensure `PIPE` and `TEXT_EMBEDDINGS` are available in the runtime before running resume script."
    )
# -----------------------------------------------------------------------

watermarking_mask = get_watermarking_mask(
    pipe.get_random_latents(),
    w_mask_shape=W_MASK_SHAPE,
    w_channel=W_CHANNEL,
    w_radius=W_RADIUS,
    device=device,
)

gt_patch = get_watermarking_pattern(
    pipe,
    w_seed=SEED,
    w_pattern=W_PATTERN,
    w_radius=W_RADIUS,
    device=device,
    strength=W_STRENGTH,
    shape=None,
)

# build model + optimizer + criterion
model = make_model(8, include_psnr_l1=INCLUDE_PSNR_L1).to(DEVICE)
model, start_epoch, best_val_loss, best_epoch = load_checkpoint(
    model, CHECKPOINT_PATH, DEVICE, opt=None
)

#  ---------------- Prepare datasets and dataloaders ----------------
file_paths, labels = discover_dataset_files(DATA_DIR)
combined = list(zip(file_paths, labels))
random.shuffle(combined)
file_paths, labels = zip(*combined)
n_val = int(len(file_paths) * VALIDATION_SPLIT)
val_paths = file_paths[:n_val]
val_labels = labels[:n_val]
print(val_labels[:50])
train_paths = file_paths[n_val:]
train_labels = labels[n_val:]

val_ds = WatermarkOnTheFlyDataset(
    val_paths,
    val_labels,
    pipe=PIPE,
    text_embeddings=TEXT_EMBEDDINGS,
    num_inference_steps=NUM_INFERENCE_STEPS,
    guidance_scale=GUIDANCE_SCALE,
    device=DEVICE,
    image_aug=None,
    include_mask_patch=INCLUDE_MASK_PATCH,
    include_psnr_l1=INCLUDE_PSNR_L1,
    watermarking_mask=watermarking_mask,
    gt_patch=gt_patch,
    psnr_return_prob=False,
)
val_ds.image_aug_prob = 1  # always apply augmentations during validation
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS
)
crit = nn.CrossEntropyLoss()

# ----------------------------------------------------------------------
# Evaluation loop (resume) with pretty printing and PSNR metrics
# ----------------------------------------------------------------------
testing_times = 5
crit = nn.CrossEntropyLoss()
total_start = time.time()


def avg(values):
    return sum(values) / len(values) if values else float("nan")


# Define attack transforms
attack_factories = {
    "clean": lambda: make_clean_aug(IMAGE_SIZE),
    "jpeg_strong": lambda: make_jpeg_aug(IMAGE_SIZE, q_low=40, q_high=60),
    "msg_app_combo": lambda: make_msg_app_combo(IMAGE_SIZE),
    "down_up": lambda: make_down_up_attack(IMAGE_SIZE, downscale_frac=0.5),
    "blur": lambda: make_blur_aug(IMAGE_SIZE),
    "random_crop": lambda: make_random_crop_attack(IMAGE_SIZE, scale=(0.5, 0.9)),
    "occlusion": lambda: make_occlusion_block(IMAGE_SIZE, box_frac=0.25),
    "geom_warp": lambda: make_geom_aug(IMAGE_SIZE),
    "train_aug_mix": lambda: make_train_image_augmentations(IMAGE_SIZE),
}

# Use existing val_loader
dataset = val_loader.dataset

os.makedirs(EVAL_RESULT_SAVE_DIR, exist_ok=True)

result_df = []

for attack_name, aug_builder in attack_factories.items():

    attack_result_dir = os.path.join(EVAL_RESULT_SAVE_DIR, attack_name)
    os.makedirs(attack_result_dir, exist_ok=True)
    dataset.image_aug = aug_builder()  # ← change augmentation in-place

    # metric trackers for this attack
    all_preds, all_gts, all_psnrs, all_l1s = [], [], [], []

    for test_i in tqdm(range(testing_times), desc=f"{attack_name} | Testing runs"):

        probs_aug, gts_aug, val_loss_aug, psnrs, l1s = eval_model_psnr_l1(
            model, val_loader, crit, DEVICE
        )

        all_preds.extend(probs_aug)
        all_gts.extend(gts_aug)
        all_psnrs.extend(psnrs)
        all_l1s.extend(l1s)

    # Compute ROC curves and best thresholds
    l1_fpr, l1_tpr, l1_thresholds = metrics.roc_curve(
        all_gts, [-x for x in all_l1s], pos_label=1
    )
    best_l1_thr = get_best_thrs(l1_fpr, l1_tpr, l1_thresholds)
    psnr_fpr, psnr_tpr, psnr_thresholds = metrics.roc_curve(
        all_gts, all_psnrs, pos_label=1
    )
    best_psnr_thr = get_best_thrs(psnr_fpr, psnr_tpr, psnr_thresholds)
    our_fpr, our_tpr, our_thresholds = metrics.roc_curve(
        all_gts, all_preds, pos_label=1
    )
    best_our_thr = get_best_thrs(our_fpr, our_tpr, our_thresholds)

    l1_auc = metrics.auc(l1_fpr, l1_tpr)
    psnr_auc = metrics.auc(psnr_fpr, psnr_tpr)
    our_auc = metrics.auc(our_fpr, our_tpr)

    # Remember the thresholds for later attack if the attack is clean
    if attack_name == "clean":
        clean_best_l1_thr = best_l1_thr
        clean_best_psnr_thr = best_psnr_thr

    # Compute final accuracy at best thresholds
    l1_preds = [1 if -l1 >= clean_best_l1_thr else 0 for l1 in all_l1s]
    psnr_preds = [1 if psnr >= clean_best_psnr_thr else 0 for psnr in all_psnrs]
    our_preds = [1 if pred >= 0.5 else 0 for pred in all_preds]
    l1_acc = sum([1 if p == gt else 0 for p, gt in zip(l1_preds, all_gts)]) / len(
        all_gts
    )
    psnr_acc = sum([1 if p == gt else 0 for p, gt in zip(psnr_preds, all_gts)]) / len(
        all_gts
    )
    our_acc = sum([1 if p == gt else 0 for p, gt in zip(our_preds, all_gts)]) / len(
        all_gts
    )
    attack_eval_results = {
        "attack_name": attack_name,
        "preds": all_preds,
        "gts": all_gts,
        "psnrs": all_psnrs,
        "l1s": all_l1s,
        "l1_acc": l1_acc,
        "l1_auc": l1_auc,
        "psnr_acc": psnr_acc,
        "psnr_auc": psnr_auc,
        "best_l1_thr": best_l1_thr,
        "best_psnr_thr": best_psnr_thr,
        "clean_best_l1_thr": clean_best_l1_thr,
        "clean_best_psnr_thr": clean_best_psnr_thr,
        "our_acc": our_acc,
        "our_auc": our_auc,
        "best_our_thr": best_our_thr,
    }
    result_df.append(
        {
            "attack": attack_name,
            "our_acc": our_acc,
            "our_auc": our_auc,
            "best_our_thr": best_our_thr,
            "l1_acc": l1_acc,
            "l1_auc": l1_auc,
            "best_l1_thr": best_l1_thr,
            "psnr_acc": psnr_acc,
            "psnr_auc": psnr_auc,
            "best_psnr_thr": best_psnr_thr,
        }
    )

    # Save results to file
    result_save_path = os.path.join(attack_result_dir, "eval_results.pt")
    torch.save(attack_eval_results, result_save_path)

    # Summary for this attack
    print("\n" + "-" * 80)
    print(f"AVERAGE ({attack_name}) over {testing_times} runs")
    print("-" * 80)
    print(f"{'Metric':<20} {'Avg Value':>18}")
    print("-" * 42)
    print(f"{'L1 Accuracy':<20} {l1_acc:>18.4f}")
    print(f"{'L1 AUROC':<20}   {l1_auc:>18.4f}")
    print(f"{'PSNR Accuracy':<20} {psnr_acc:>18.4f}")
    print(f"{'PSNR AUROC':<20} {psnr_auc:>18.4f}")
    print(f"{'Our Accuracy':<20} {our_acc:>18.4f}")
    print(f"{'Our AUROC':<20} {our_auc:>18.4f}")
    print(f"{'Best L1 Thr':<20} {best_l1_thr:>18.4f}")
    print(f"{'Best PSNR Thr':<20} {best_psnr_thr:>18.4f}")
    print(f"{'Best Our Thr':<20} {best_our_thr:>18.4f}")
    # threshold used.
    print(
        f"Thresholds used for Acc: L1={clean_best_l1_thr:.4f}, PSNR={clean_best_psnr_thr:.4f}"
    )
    print("-" * 42)
    print("#" * 80)


print("\n" + "=" * 80)
print("All evaluations complete.")
print("=" * 80)

df_results = pd.DataFrame(result_df)
display(df_results)
# save results to CSV
csv_path = os.path.join(EVAL_RESULT_SAVE_DIR, "attack_eval_summary.csv")
df_results.to_csv(csv_path, index=False)
print(f"Saved summary results to {csv_path}")

Keyword arguments {'verbose': False} are not expected by InversableStableDiffusionPipeline and will be ignored.


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!


Using PNSRL1 wrapper for the model.
Loading checkpoint: verifier_dataset_stablediff_octoweb_meta_verifier_300.pth
Restored model and optimizer. Resuming from epoch 98
(1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0)


clean | Testing runs: 100%|██████████| 5/5 [15:27<00:00, 185.45s/it]


...


--------------------------------------------------------------------------------
AVERAGE (geom_warp) over 5 runs
--------------------------------------------------------------------------------
Metric                        Avg Value
------------------------------------------
L1 Accuracy                      0.5787
L1 AUROC                           0.9724
PSNR Accuracy                    0.5373
PSNR AUROC                       0.9054
Our Accuracy                     0.8093
Our AUROC                        0.9558
Best L1 Thr                    -52.3438
Best PSNR Thr                  -19.4021
Best Our Thr                     0.1860
Thresholds used for Acc: L1=-45.7812, PSNR=-17.7885
------------------------------------------
################################################################################


train_aug_mix | Testing runs: 100%|██████████| 5/5 [26:18<00:00, 315.63s/it]


--------------------------------------------------------------------------------
AVERAGE (train_aug_mix) over 5 runs
--------------------------------------------------------------------------------
Metric                        Avg Value
------------------------------------------
L1 Accuracy                      0.5293
L1 AUROC                           0.8658
PSNR Accuracy                    0.5187
PSNR AUROC                       0.7851
Our Accuracy                     0.6427
Our AUROC                        0.8627
Best L1 Thr                    -53.9062
Best PSNR Thr                  -19.3640
Best Our Thr                     0.1347
Thresholds used for Acc: L1=-45.7812, PSNR=-17.7885
------------------------------------------
################################################################################

All evaluations complete.


,attack,our_acc,our_auc,best_our_thr,l1_acc,l1_auc,best_l1_thr,psnr_acc,psnr_auc,best_psnr_thr
0,clean,0.966667,0.999643,0.934340,0.986667,0.998930,-45.78125,0.980000,0.995899,-17.788549
1,jpeg_strong,0.754667,0.942685,0.213952,0.644000,0.952252,-52.31250,0.609333,0.888707,-19.276882
2,msg_app_combo,0.509333,0.836363,0.036856,0.480000,0.870198,-56.78125,0.476000,0.728978,-20.411182
3,down_up,0.900000,0.974862,0.428564,0.733333,0.985648,-53.75000,0.573333,0.943305,-19.649616
4,blur,0.669333,0.839394,0.098315,0.500000,0.886311,-55.25000,0.478667,0.800121,-20.251005
5,random_crop,0.828000,0.957347,0.175174,0.624000,0.975842,-52.21875,0.560000,0.947627,-19.271465
6,occlusion,0.978667,0.998980,0.725949,0.973333,0.998224,-48.53125,0.946667,0.994138,-18.205666
7,geom_warp,0.809333,0.955814,0.185950,0.578667,0.972430,-52.34375,0.537333,0.905438,-19.402058
8,train_aug_mix,0.642667,0.862749,0.134686,0.529333,0.865819,-53.90625,0.518667,0.785053,-19.363953


Saved summary results to ./eval_results/our_improved_meta\attack_eval_summary.csv


In [ ]:
# Keyword arguments {'verbose': False} are not expected by InversableStableDiffusionPipeline and will be ignored.
# Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]An error occurred while trying to fetch C:\Users\mike8\.cache\huggingface\hub\models--stabilityai--stable-diffusion-2-1-base\snapshots\1f758383196d38df1dfe523ddb1030f2bfab7741\vae: Error no file named diffusion_pytorch_model.safetensors found in directory C:\Users\mike8\.cache\huggingface\hub\models--stabilityai--stable-diffusion-2-1-base\snapshots\1f758383196d38df1dfe523ddb1030f2bfab7741\vae.
# Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
# An error occurred while trying to fetch C:\Users\mike8\.cache\huggingface\hub\models--stabilityai--stable-diffusion-2-1-base\snapshots\1f758383196d38df1dfe523ddb1030f2bfab7741\unet: Error no file named diffusion_pytorch_model.safetensors found in directory C:\Users\mike8\.cache\huggingface\hub\models--stabilityai--stable-diffusion-2-1-base\snapshots\1f758383196d38df1dfe523ddb1030f2bfab7741\unet.
# Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
# Loading pipeline components...:  60%|██████    | 3/5 [00:00<00:00, 20.70it/s]`torch_dtype` is deprecated! Use `dtype` instead!
# Loading pipeline components...: 100%|██████████| 5/5 [00:00<00:00, 12.22it/s]
# Using PNSR wrapper for the model.
# Loading checkpoint: PNSR_AND_PATTERN.pth
# Warning: couldn't fully load optimizer state: 'NoneType' object has no attribute 'load_state_dict'
# Restored model and optimizer. Resuming from epoch 133

# ################################################################################
# # ATTACK: clean
# ################################################################################
# ================================================================================
# [clean] Test   1/  5    Time elapsed: 0:00:00
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.9333
# AUROC                              0.9918
# PSNR_Acc                         0.9533
# PSNR_AUC                         0.9948
# PSNR_Acc_2                       0.9533
# PSNR_AUC_2                       0.9557
# Val Loss                         0.2038
# -----------------------------------------
# Iter time: 386.5s
# ================================================================================
# ================================================================================
# [clean] Test   2/  5    Time elapsed: 0:06:26
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.9333
# AUROC                              0.9918
# PSNR_Acc                         0.9533
# PSNR_AUC                         0.9948
# PSNR_Acc_2                       0.9533
# PSNR_AUC_2                       0.9557
# Val Loss                         0.2038
# -----------------------------------------
# Iter time: 400.4s
# ================================================================================
# ================================================================================
# [clean] Test   3/  5    Time elapsed: 0:13:07
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.9333
# AUROC                              0.9918
# PSNR_Acc                         0.9533
# PSNR_AUC                         0.9948
# PSNR_Acc_2                       0.9533
# PSNR_AUC_2                       0.9557
# Val Loss                         0.2038
# -----------------------------------------
# Iter time: 389.8s
# ================================================================================
# ================================================================================
# [clean] Test   4/  5    Time elapsed: 0:19:36
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.9333
# AUROC                              0.9918
# PSNR_Acc                         0.9533
# PSNR_AUC                         0.9948
# PSNR_Acc_2                       0.9533
# PSNR_AUC_2                       0.9557
# Val Loss                         0.2038
# -----------------------------------------
# Iter time: 382.4s
# ================================================================================
# ================================================================================
# [clean] Test   5/  5    Time elapsed: 0:25:59
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.9333
# AUROC                              0.9918
# PSNR_Acc                         0.9533
# PSNR_AUC                         0.9948
# PSNR_Acc_2                       0.9533
# PSNR_AUC_2                       0.9557
# Val Loss                         0.2038
# -----------------------------------------
# Iter time: 383.4s
# ================================================================================

# --------------------------------------------------------------------------------
# AVERAGE (clean) over 5 runs
# --------------------------------------------------------------------------------
# Metric                        Avg Value
# ------------------------------------------
# Accuracy                         0.9333
# AUROC                              0.9918
# PSNR_Acc                         0.9533
# PSNR_AUC                         0.9948
# ------------------------------------------
# ################################################################################

# ################################################################################
# # ATTACK: jpeg_strong
# ################################################################################
# ================================================================================
# [jpeg_strong] Test   1/  5    Time elapsed: 0:32:22
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.7200
# AUROC                              0.8272
# PSNR_Acc                         0.6067
# PSNR_AUC                         0.8568
# PSNR_Acc_2                       0.5800
# PSNR_AUC_2                       0.6013
# Val Loss                         0.5367
# -----------------------------------------
# Iter time: 382.1s
# ================================================================================
# ================================================================================
# [jpeg_strong] Test   2/  5    Time elapsed: 0:38:44
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.7867
# AUROC                              0.8581
# PSNR_Acc                         0.6133
# PSNR_AUC                         0.8549
# PSNR_Acc_2                       0.6000
# PSNR_AUC_2                       0.6203
# Val Loss                         0.5246
# -----------------------------------------
# Iter time: 377.9s
# ================================================================================
# ================================================================================
# [jpeg_strong] Test   3/  5    Time elapsed: 0:45:02
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.7533
# AUROC                              0.8246
# PSNR_Acc                         0.6067
# PSNR_AUC                         0.8367
# PSNR_Acc_2                       0.5933
# PSNR_AUC_2                       0.6139
# Val Loss                         0.5626
# -----------------------------------------
# Iter time: 381.3s
# ================================================================================
# ================================================================================
# [jpeg_strong] Test   4/  5    Time elapsed: 0:51:24
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.7333
# AUROC                              0.8174
# PSNR_Acc                         0.5933
# PSNR_AUC                         0.8429
# PSNR_Acc_2                       0.5667
# PSNR_AUC_2                       0.5879
# Val Loss                         0.5914
# -----------------------------------------
# Iter time: 383.4s
# ================================================================================
# ================================================================================
# [jpeg_strong] Test   5/  5    Time elapsed: 0:57:47
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.7400
# AUROC                              0.8214
# PSNR_Acc                         0.6133
# PSNR_AUC                         0.8529
# PSNR_Acc_2                       0.6000
# PSNR_AUC_2                       0.6203
# Val Loss                         0.5793
# -----------------------------------------
# Iter time: 378.6s
# ================================================================================

# --------------------------------------------------------------------------------
# AVERAGE (jpeg_strong) over 5 runs
# --------------------------------------------------------------------------------
# Metric                        Avg Value
# ------------------------------------------
# Accuracy                         0.7467
# AUROC                            0.8297
# PSNR_Acc                         0.6067
# PSNR_AUC                         0.8489
# ------------------------------------------
# ################################################################################

# ################################################################################
# # ATTACK: msg_app_combo
# ################################################################################
# ================================================================================
# [msg_app_combo] Test   1/  5    Time elapsed: 1:04:05
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.5800
# AUROC                              0.6976
# PSNR_Acc                         0.4733
# PSNR_AUC                         0.6969
# PSNR_Acc_2                       0.4733
# PSNR_AUC_2                       0.5000
# Val Loss                         0.9263
# -----------------------------------------
# Iter time: 381.9s
# ================================================================================
# ================================================================================
# [msg_app_combo] Test   2/  5    Time elapsed: 1:10:27
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.5400
# AUROC                              0.6639
# PSNR_Acc                         0.4733
# PSNR_AUC                         0.6711
# PSNR_Acc_2                       0.4733
# PSNR_AUC_2                       0.5000
# Val Loss                         0.9382
# -----------------------------------------
# Iter time: 384.8s
# ================================================================================
# ================================================================================
# [msg_app_combo] Test   3/  5    Time elapsed: 1:16:52
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.5800
# AUROC                              0.6568
# PSNR_Acc                         0.4733
# PSNR_AUC                         0.6707
# PSNR_Acc_2                       0.4733
# PSNR_AUC_2                       0.5000
# Val Loss                         0.9965
# -----------------------------------------
# Iter time: 390.1s
# ================================================================================
# ================================================================================
# [msg_app_combo] Test   4/  5    Time elapsed: 1:23:22
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.5667
# AUROC                              0.7058
# PSNR_Acc                         0.4733
# PSNR_AUC                         0.6802
# PSNR_Acc_2                       0.4733
# PSNR_AUC_2                       0.5000
# Val Loss                         0.9575
# -----------------------------------------
# Iter time: 382.4s
# ================================================================================
# ================================================================================
# [msg_app_combo] Test   5/  5    Time elapsed: 1:29:45
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.5533
# AUROC                              0.6784
# PSNR_Acc                         0.4733
# PSNR_AUC                         0.6654
# PSNR_Acc_2                       0.4733
# PSNR_AUC_2                       0.5000
# Val Loss                         0.9227
# -----------------------------------------
# Iter time: 384.3s
# ================================================================================

# --------------------------------------------------------------------------------
# AVERAGE (msg_app_combo) over 5 runs
# --------------------------------------------------------------------------------
# Metric                        Avg Value
# ------------------------------------------
# Accuracy                         0.5640
# AUROC                              0.6805
# PSNR_Acc                         0.4733
# PSNR_AUC                         0.6768
# ------------------------------------------
# ################################################################################

# ################################################################################
# # ATTACK: down_up
# ################################################################################
# ================================================================================
# [down_up] Test   1/  5    Time elapsed: 1:36:09
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.8133
# AUROC                              0.8645
# PSNR_Acc                         0.5533
# PSNR_AUC                         0.9030
# PSNR_Acc_2                       0.5533
# PSNR_AUC_2                       0.5759
# Val Loss                         0.5176
# -----------------------------------------
# Iter time: 382.2s
# ================================================================================
# ================================================================================
# [down_up] Test   2/  5    Time elapsed: 1:42:31
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.8133
# AUROC                              0.8645
# PSNR_Acc                         0.5533
# PSNR_AUC                         0.9030
# PSNR_Acc_2                       0.5533
# PSNR_AUC_2                       0.5759
# Val Loss                         0.5176
# -----------------------------------------
# Iter time: 380.7s
# ================================================================================
# ================================================================================
# [down_up] Test   3/  5    Time elapsed: 1:48:52
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.8133
# AUROC                              0.8645
# PSNR_Acc                         0.5533
# PSNR_AUC                         0.9030
# PSNR_Acc_2                       0.5533
# PSNR_AUC_2                       0.5759
# Val Loss                         0.5176
# -----------------------------------------
# Iter time: 380.6s
# ================================================================================
# ================================================================================
# [down_up] Test   4/  5    Time elapsed: 1:55:13
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.8133
# AUROC                              0.8645
# PSNR_Acc                         0.5533
# PSNR_AUC                         0.9030
# PSNR_Acc_2                       0.5533
# PSNR_AUC_2                       0.5759
# Val Loss                         0.5176
# -----------------------------------------
# Iter time: 381.1s
# ================================================================================
# ================================================================================
# [down_up] Test   5/  5    Time elapsed: 2:01:34
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.8133
# AUROC                              0.8645
# PSNR_Acc                         0.5533
# PSNR_AUC                         0.9030
# PSNR_Acc_2                       0.5533
# PSNR_AUC_2                       0.5759
# Val Loss                         0.5176
# -----------------------------------------
# Iter time: 378.4s
# ================================================================================

# --------------------------------------------------------------------------------
# AVERAGE (down_up) over 5 runs
# --------------------------------------------------------------------------------
# Metric                        Avg Value
# ------------------------------------------
# Accuracy                         0.8133
# AUROC                              0.8645
# PSNR_Acc                         0.5533
# PSNR_AUC                         0.9030
# ------------------------------------------
# ################################################################################

# ################################################################################
# # ATTACK: blur
# ################################################################################
# ================================================================================
# [blur] Test   1/  5    Time elapsed: 2:07:52
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.6133
# AUROC                              0.6884
# PSNR_Acc                         0.4733
# PSNR_AUC                         0.7106
# PSNR_Acc_2                       0.4733
# PSNR_AUC_2                       0.5000
# Val Loss                         0.9185
# -----------------------------------------
# Iter time: 384.6s
# ================================================================================
# ================================================================================
# [blur] Test   2/  5    Time elapsed: 2:14:17
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.6400
# AUROC                              0.7294
# PSNR_Acc                         0.4733
# PSNR_AUC                         0.7427
# PSNR_Acc_2                       0.4733
# PSNR_AUC_2                       0.5000
# Val Loss                         0.8862
# -----------------------------------------
# Iter time: 392.0s
# ================================================================================
# ================================================================================
# [blur] Test   3/  5    Time elapsed: 2:20:49
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.5933
# AUROC                              0.6698
# PSNR_Acc                         0.4800
# PSNR_AUC                         0.7057
# PSNR_Acc_2                       0.4800
# PSNR_AUC_2                       0.5063
# Val Loss                         0.9685
# -----------------------------------------
# Iter time: 397.7s
# ================================================================================
# ================================================================================
# [blur] Test   4/  5    Time elapsed: 2:27:26
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.6400
# AUROC                              0.7602
# PSNR_Acc                         0.4733
# PSNR_AUC                         0.7889
# PSNR_Acc_2                       0.4733
# PSNR_AUC_2                       0.5000
# Val Loss                         0.8068
# -----------------------------------------
# Iter time: 394.1s
# ================================================================================
# ================================================================================
# [blur] Test   5/  5    Time elapsed: 2:34:00
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.6200
# AUROC                              0.6655
# PSNR_Acc                         0.4733
# PSNR_AUC                         0.7073
# PSNR_Acc_2                       0.4733
# PSNR_AUC_2                       0.5000
# Val Loss                         0.9983
# -----------------------------------------
# Iter time: 403.1s
# ================================================================================

# --------------------------------------------------------------------------------
# AVERAGE (blur) over 5 runs
# --------------------------------------------------------------------------------
# Metric                        Avg Value
# ------------------------------------------
# Accuracy                         0.6213
# AUROC                              0.7027
# PSNR_Acc                         0.4747
# PSNR_AUC                         0.7310
# ------------------------------------------
# ################################################################################

# ################################################################################
# # ATTACK: random_crop
# ################################################################################
# ================================================================================
# [random_crop] Test   1/  5    Time elapsed: 2:40:43
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.7667
# AUROC                              0.8750
# PSNR_Acc                         0.5867
# PSNR_AUC                         0.9071
# PSNR_Acc_2                       0.5600
# PSNR_AUC_2                       0.5823
# Val Loss                         0.4597
# -----------------------------------------
# Iter time: 398.8s
# ================================================================================
# ================================================================================
# [random_crop] Test   2/  5    Time elapsed: 2:47:22
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.7933
# AUROC                              0.8896
# PSNR_Acc                         0.5667
# PSNR_AUC                         0.8952
# PSNR_Acc_2                       0.5733
# PSNR_AUC_2                       0.5949
# Val Loss                         0.4375
# -----------------------------------------
# Iter time: 391.3s
# ================================================================================
# ================================================================================
# [random_crop] Test   3/  5    Time elapsed: 2:53:54
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.8333
# AUROC                              0.9289
# PSNR_Acc                         0.6200
# PSNR_AUC                         0.9458
# PSNR_Acc_2                       0.5400
# PSNR_AUC_2                       0.5633
# Val Loss                         0.3648
# -----------------------------------------
# Iter time: 399.6s
# ================================================================================
# ================================================================================
# [random_crop] Test   4/  5    Time elapsed: 3:00:33
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.8267
# AUROC                              0.8768
# PSNR_Acc                         0.5800
# PSNR_AUC                         0.9230
# PSNR_Acc_2                       0.6200
# PSNR_AUC_2                       0.6392
# Val Loss                         0.5174
# -----------------------------------------
# Iter time: 397.7s
# ================================================================================
# ================================================================================
# [random_crop] Test   5/  5    Time elapsed: 3:07:11
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.8200
# AUROC                              0.8855
# PSNR_Acc                         0.5667
# PSNR_AUC                         0.9330
# PSNR_Acc_2                       0.5800
# PSNR_AUC_2                       0.6013
# Val Loss                         0.4716
# -----------------------------------------
# Iter time: 395.3s
# ================================================================================

# --------------------------------------------------------------------------------
# AVERAGE (random_crop) over 5 runs
# --------------------------------------------------------------------------------
# Metric                        Avg Value
# ------------------------------------------
# Accuracy                         0.8080
# AUROC                              0.8912
# PSNR_Acc                         0.5840
# PSNR_AUC                         0.9208
# ------------------------------------------
# ################################################################################

# ################################################################################
# # ATTACK: occlusion
# ################################################################################
# ================================================================================
# [occlusion] Test   1/  5    Time elapsed: 3:13:46
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.9200
# AUROC                              0.9781
# PSNR_Acc                         0.9333
# PSNR_AUC                         0.9950
# PSNR_Acc_2                       0.9200
# PSNR_AUC_2                       0.9241
# Val Loss                         0.2142
# -----------------------------------------
# Iter time: 383.1s
# ================================================================================
# ================================================================================
# [occlusion] Test   2/  5    Time elapsed: 3:20:09
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.9133
# AUROC                              0.9734
# PSNR_Acc                         0.9400
# PSNR_AUC                         0.9939
# PSNR_Acc_2                       0.9267
# PSNR_AUC_2                       0.9304
# Val Loss                         0.2142
# -----------------------------------------
# Iter time: 388.5s
# ================================================================================
# ================================================================================
# [occlusion] Test   3/  5    Time elapsed: 3:26:38
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.9200
# AUROC                              0.9829
# PSNR_Acc                         0.9133
# PSNR_AUC                         0.9939
# PSNR_Acc_2                       0.9200
# PSNR_AUC_2                       0.9241
# Val Loss                         0.1938
# -----------------------------------------
# Iter time: 375.9s
# ================================================================================
# ================================================================================
# [occlusion] Test   4/  5    Time elapsed: 3:32:54
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.9267
# AUROC                              0.9761
# PSNR_Acc                         0.9400
# PSNR_AUC                         0.9927
# PSNR_Acc_2                       0.9333
# PSNR_AUC_2                       0.9367
# Val Loss                         0.2114
# -----------------------------------------
# Iter time: 383.8s
# ================================================================================
# ================================================================================
# [occlusion] Test   5/  5    Time elapsed: 3:39:17
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.9333
# AUROC                              0.9766
# PSNR_Acc                         0.9200
# PSNR_AUC                         0.9927
# PSNR_Acc_2                       0.9333
# PSNR_AUC_2                       0.9367
# Val Loss                         0.2063
# -----------------------------------------
# Iter time: 383.3s
# ================================================================================

# --------------------------------------------------------------------------------
# AVERAGE (occlusion) over 5 runs
# --------------------------------------------------------------------------------
# Metric                        Avg Value
# ------------------------------------------
# Accuracy                         0.9227
# AUROC                              0.9774
# PSNR_Acc                         0.9293
# PSNR_AUC                         0.9937
# ------------------------------------------
# ################################################################################

# ################################################################################
# # ATTACK: geom_warp
# ################################################################################
# ================================================================================
# [geom_warp] Test   1/  5    Time elapsed: 3:45:41
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.8133
# AUROC                              0.8822
# PSNR_Acc                         0.5533
# PSNR_AUC                         0.8994
# PSNR_Acc_2                       0.5333
# PSNR_AUC_2                       0.5570
# Val Loss                         0.4319
# -----------------------------------------
# Iter time: 382.8s
# ================================================================================
# ================================================================================
# [geom_warp] Test   2/  5    Time elapsed: 3:52:03
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.8400
# AUROC                              0.9182
# PSNR_Acc                         0.5600
# PSNR_AUC                         0.9089
# PSNR_Acc_2                       0.5600
# PSNR_AUC_2                       0.5823
# Val Loss                         0.3926
# -----------------------------------------
# Iter time: 383.4s
# ================================================================================
# ================================================================================
# [geom_warp] Test   3/  5    Time elapsed: 3:58:27
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.7867
# AUROC                              0.8763
# PSNR_Acc                         0.6000
# PSNR_AUC                         0.8863
# PSNR_Acc_2                       0.5333
# PSNR_AUC_2                       0.5570
# Val Loss                         0.4396
# -----------------------------------------
# Iter time: 385.9s
# ================================================================================
# ================================================================================
# [geom_warp] Test   4/  5    Time elapsed: 4:04:53
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.7733
# AUROC                              0.8707
# PSNR_Acc                         0.5333
# PSNR_AUC                         0.8727
# PSNR_Acc_2                       0.5467
# PSNR_AUC_2                       0.5696
# Val Loss                         0.4593
# -----------------------------------------
# Iter time: 386.1s
# ================================================================================
# ================================================================================
# [geom_warp] Test   5/  5    Time elapsed: 4:11:19
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.7733
# AUROC                              0.8636
# PSNR_Acc                         0.5667
# PSNR_AUC                         0.8706
# PSNR_Acc_2                       0.5733
# PSNR_AUC_2                       0.5949
# Val Loss                         0.4715
# -----------------------------------------
# Iter time: 384.3s
# ================================================================================

# --------------------------------------------------------------------------------
# AVERAGE (geom_warp) over 5 runs
# --------------------------------------------------------------------------------
# Metric                        Avg Value
# ------------------------------------------
# Accuracy                         0.7973
# AUROC                              0.8822
# PSNR_Acc                         0.5627
# PSNR_AUC                         0.8876
# ------------------------------------------
# ################################################################################

# ################################################################################
# # ATTACK: train_aug_mix
# ################################################################################
# ================================================================================
# [train_aug_mix] Test   1/  5    Time elapsed: 4:17:43
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.7067
# AUROC                              0.7675
# PSNR_Acc                         0.5133
# PSNR_AUC                         0.7506
# PSNR_Acc_2                       0.5533
# PSNR_AUC_2                       0.5759
# Val Loss                         0.6584
# -----------------------------------------
# Iter time: 387.3s
# ================================================================================
# ================================================================================
# [train_aug_mix] Test   2/  5    Time elapsed: 4:24:10
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.7067
# AUROC                              0.7595
# PSNR_Acc                         0.5333
# PSNR_AUC                         0.7479
# PSNR_Acc_2                       0.5200
# PSNR_AUC_2                       0.5443
# Val Loss                         0.6608
# -----------------------------------------
# Iter time: 390.2s
# ================================================================================
# ================================================================================
# [train_aug_mix] Test   3/  5    Time elapsed: 4:30:41
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.7067
# AUROC                              0.8058
# PSNR_Acc                         0.5533
# PSNR_AUC                         0.8440
# PSNR_Acc_2                       0.5733
# PSNR_AUC_2                       0.5949
# Val Loss                         0.5654
# -----------------------------------------
# Iter time: 386.5s
# ================================================================================
# ================================================================================
# [train_aug_mix] Test   4/  5    Time elapsed: 4:37:07
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.7533
# AUROC                              0.8087
# PSNR_Acc                         0.5200
# PSNR_AUC                         0.7873
# PSNR_Acc_2                       0.5600
# PSNR_AUC_2                       0.5823
# Val Loss                         0.5677
# -----------------------------------------
# Iter time: 388.5s
# ================================================================================
# ================================================================================
# [train_aug_mix] Test   5/  5    Time elapsed: 4:43:36
# --------------------------------------------------------------------------------
#                                                      Metric                            Value
# -----------------------------------------
# Accuracy                         0.7667
# AUROC                              0.8214
# PSNR_Acc                         0.5667
# PSNR_AUC                         0.8540
# PSNR_Acc_2                       0.5400
# PSNR_AUC_2                       0.5633
# Val Loss                         0.5868
# -----------------------------------------
# Iter time: 381.9s
# ================================================================================

# --------------------------------------------------------------------------------
# AVERAGE (train_aug_mix) over 5 runs
# --------------------------------------------------------------------------------
# Metric                        Avg Value
# ------------------------------------------
# Accuracy                         0.7280
# AUROC                              0.7926
# PSNR_Acc                         0.5373
# PSNR_AUC                         0.7968
# ------------------------------------------
# ################################################################################

# ================================================================================
# All evaluations complete.
# ================================================================================

In [3]:
# LR: 2.000e-04
# --------------------------------------------------------------------------------
                                                        
# Metric                            Train          Val (AUG)       Val (NO-AUG)
# --------------------------------------------------------------------------------
# Loss                             0.2427             0.3991             0.2483
# Accuracy                             --             0.8467             0.9267
# AUROC                                --             0.9162             0.9920
# --------------------------------------------------------------------------------
# Epoch time: 1289.2s (train: 925.8s). Cumulative: 10:22:51
# Best no-aug val loss so far: 0.225697 (epoch 116)
# No improvement (current no-aug val loss 0.248273)
# ================================================================================
# Epoch 132/200    Time elapsed: 10:22:51
# LR: 2.000e-04
# --------------------------------------------------------------------------------

In [4]:
# Keyword arguments {'verbose': False} are not expected by InversableStableDiffusionPipeline and will be ignored.
# Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]`torch_dtype` is deprecated! Use `dtype` instead!
# Loading pipeline components...:  20%|██        | 1/5 [00:00<00:00,  4.91it/s]An error occurred while trying to fetch C:\Users\mike8\.cache\huggingface\hub\models--stabilityai--stable-diffusion-2-1-base\snapshots\1f758383196d38df1dfe523ddb1030f2bfab7741\unet: Error no file named diffusion_pytorch_model.safetensors found in directory C:\Users\mike8\.cache\huggingface\hub\models--stabilityai--stable-diffusion-2-1-base\snapshots\1f758383196d38df1dfe523ddb1030f2bfab7741\unet.
# Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
# Loading pipeline components...:  60%|██████    | 3/5 [00:00<00:00,  9.89it/s]An error occurred while trying to fetch C:\Users\mike8\.cache\huggingface\hub\models--stabilityai--stable-diffusion-2-1-base\snapshots\1f758383196d38df1dfe523ddb1030f2bfab7741\vae: Error no file named diffusion_pytorch_model.safetensors found in directory C:\Users\mike8\.cache\huggingface\hub\models--stabilityai--stable-diffusion-2-1-base\snapshots\1f758383196d38df1dfe523ddb1030f2bfab7741\vae.
# Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
# Loading pipeline components...: 100%|██████████| 5/5 [00:00<00:00, 13.41it/s]
# Using PNSR wrapper for the model.
# Loading checkpoint: PNSR_AND_PATTERN.pth
# Warning: couldn't fully load optimizer state: 'NoneType' object has no attribute 'load_state_dict'
# Restored model and optimizer. Resuming from epoch 133

# ################################################################################
# # ATTACK: clean
# ################################################################################
# ================================================================================
# [clean] Test   1/  5    Time elapsed: 0:00:00
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.9333
# AUROC                              0.9918
# PSNR_Acc                         0.9533
# PSNR_AUC                         0.9948
# Val Loss                         0.2038
# -----------------------------------------
# Iter time: 159.6s
# ================================================================================
# ================================================================================
# [clean] Test   2/  5    Time elapsed: 0:02:39
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.9333
# AUROC                              0.9918
# PSNR_Acc                         0.9533
# PSNR_AUC                         0.9948
# Val Loss                         0.2038
# -----------------------------------------
# Iter time: 161.3s
# ================================================================================
# ================================================================================
# [clean] Test   3/  5    Time elapsed: 0:05:20
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.9333
# AUROC                              0.9918
# PSNR_Acc                         0.9533
# PSNR_AUC                         0.9948
# Val Loss                         0.2038
# -----------------------------------------
# Iter time: 159.0s
# ================================================================================
# ================================================================================
# [clean] Test   4/  5    Time elapsed: 0:07:59
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.9333
# AUROC                              0.9918
# PSNR_Acc                         0.9533
# PSNR_AUC                         0.9948
# Val Loss                         0.2038
# -----------------------------------------
# Iter time: 165.4s
# ================================================================================
# ================================================================================
# [clean] Test   5/  5    Time elapsed: 0:10:45
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.9333
# AUROC                              0.9918
# PSNR_Acc                         0.9533
# PSNR_AUC                         0.9948
# Val Loss                         0.2038
# -----------------------------------------
# Iter time: 172.6s
# ================================================================================

# --------------------------------------------------------------------------------
# AVERAGE (clean) over 5 runs
# --------------------------------------------------------------------------------
# Metric                        Avg Value
# ------------------------------------------
# Accuracy                         0.9333
# AUROC                              0.9918
# PSNR_Acc                         0.9533
# PSNR_AUC                         0.9948
# ------------------------------------------
# ################################################################################

# ################################################################################
# # ATTACK: jpeg_strong
# ################################################################################
# ================================================================================
# [jpeg_strong] Test   1/  5    Time elapsed: 0:13:37
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.7667
# AUROC                              0.8410
# PSNR_Acc                         0.5800
# PSNR_AUC                         0.8649
# Val Loss                         0.5454
# -----------------------------------------
# Iter time: 172.1s
# ================================================================================
# ================================================================================
# [jpeg_strong] Test   2/  5    Time elapsed: 0:16:30
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.7200
# AUROC                              0.8249
# PSNR_Acc                         0.5933
# PSNR_AUC                         0.8504
# Val Loss                         0.5720
# -----------------------------------------
# Iter time: 170.8s
# ================================================================================
# ================================================================================
# [jpeg_strong] Test   3/  5    Time elapsed: 0:19:20
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.7533
# AUROC                              0.8005
# PSNR_Acc                         0.6067
# PSNR_AUC                         0.8518
# Val Loss                         0.6358
# -----------------------------------------
# Iter time: 173.3s
# ================================================================================
# ================================================================================
# [jpeg_strong] Test   4/  5    Time elapsed: 0:22:14
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.7467
# AUROC                              0.8315
# PSNR_Acc                         0.6000
# PSNR_AUC                         0.8299
# Val Loss                         0.5521
# -----------------------------------------
# Iter time: 177.6s
# ================================================================================
# ================================================================================
# [jpeg_strong] Test   5/  5    Time elapsed: 0:25:11
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.7667
# AUROC                              0.8365
# PSNR_Acc                         0.6067
# PSNR_AUC                         0.8624
# Val Loss                         0.5804
# -----------------------------------------
# Iter time: 181.7s
# ================================================================================

# --------------------------------------------------------------------------------
# AVERAGE (jpeg_strong) over 5 runs
# --------------------------------------------------------------------------------
# Metric                        Avg Value
# ------------------------------------------
# Accuracy                         0.7507
# AUROC                              0.8269
# PSNR_Acc                         0.5973
# PSNR_AUC                         0.8519
# ------------------------------------------
# ################################################################################

# ################################################################################
# # ATTACK: msg_app_combo
# ################################################################################
# ================================================================================
# [msg_app_combo] Test   1/  5    Time elapsed: 0:28:13
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.5133
# AUROC                              0.6889
# PSNR_Acc                         0.4733
# PSNR_AUC                         0.6673
# Val Loss                         0.9425
# -----------------------------------------
# Iter time: 176.9s
# ================================================================================
# ================================================================================
# [msg_app_combo] Test   2/  5    Time elapsed: 0:31:10
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.5533
# AUROC                              0.6909
# PSNR_Acc                         0.4733
# PSNR_AUC                         0.6823
# Val Loss                         0.9947
# -----------------------------------------
# Iter time: 193.3s
# ================================================================================
# ================================================================================
# [msg_app_combo] Test   3/  5    Time elapsed: 0:34:23
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.5733
# AUROC                              0.6987
# PSNR_Acc                         0.4733
# PSNR_AUC                         0.6903
# Val Loss                         0.9217
# -----------------------------------------
# Iter time: 191.3s
# ================================================================================
# ================================================================================
# [msg_app_combo] Test   4/  5    Time elapsed: 0:37:34
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.5733
# AUROC                              0.6725
# PSNR_Acc                         0.4733
# PSNR_AUC                         0.6816
# Val Loss                         0.9867
# -----------------------------------------
# Iter time: 191.0s
# ================================================================================
# ================================================================================
# [msg_app_combo] Test   5/  5    Time elapsed: 0:40:45
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.5867
# AUROC                              0.6288
# PSNR_Acc                         0.4733
# PSNR_AUC                         0.6523
# Val Loss                         1.0093
# -----------------------------------------
# Iter time: 199.7s
# ================================================================================

# --------------------------------------------------------------------------------
# AVERAGE (msg_app_combo) over 5 runs
# --------------------------------------------------------------------------------
# Metric                        Avg Value
# ------------------------------------------
# Accuracy                         0.5600
# AUROC                              0.6759
# PSNR_Acc                         0.4733
# PSNR_AUC                         0.6748
# ------------------------------------------
# ################################################################################

# ################################################################################
# # ATTACK: down_up
# ################################################################################
# ================================================================================
# [down_up] Test   1/  5    Time elapsed: 0:44:05
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.8133
# AUROC                              0.8645
# PSNR_Acc                         0.5533
# PSNR_AUC                         0.9030
# Val Loss                         0.5176
# -----------------------------------------
# Iter time: 179.3s
# ================================================================================
# ================================================================================
# [down_up] Test   2/  5    Time elapsed: 0:47:04
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.8133
# AUROC                              0.8645
# PSNR_Acc                         0.5533
# PSNR_AUC                         0.9030
# Val Loss                         0.5176
# -----------------------------------------
# Iter time: 164.5s
# ================================================================================
# ================================================================================
# [down_up] Test   3/  5    Time elapsed: 0:49:49
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.8133
# AUROC                              0.8645
# PSNR_Acc                         0.5533
# PSNR_AUC                         0.9030
# Val Loss                         0.5176
# -----------------------------------------
# Iter time: 172.2s
# ================================================================================
# ================================================================================
# [down_up] Test   4/  5    Time elapsed: 0:52:41
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.8133
# AUROC                              0.8645
# PSNR_Acc                         0.5533
# PSNR_AUC                         0.9030
# Val Loss                         0.5176
# -----------------------------------------
# Iter time: 183.3s
# ================================================================================
# ================================================================================
# [down_up] Test   5/  5    Time elapsed: 0:55:44
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.8133
# AUROC                              0.8645
# PSNR_Acc                         0.5533
# PSNR_AUC                         0.9030
# Val Loss                         0.5176
# -----------------------------------------
# Iter time: 184.1s
# ================================================================================

# --------------------------------------------------------------------------------
# AVERAGE (down_up) over 5 runs
# --------------------------------------------------------------------------------
# Metric                        Avg Value
# ------------------------------------------
# Accuracy                         0.8133
# AUROC                              0.8645
# PSNR_Acc                         0.5533
# PSNR_AUC                         0.9030
# ------------------------------------------
# ################################################################################

# ################################################################################
# # ATTACK: blur
# ################################################################################
# ================================================================================
# [blur] Test   1/  5    Time elapsed: 0:58:48
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.5867
# AUROC                              0.7135
# PSNR_Acc                         0.4733
# PSNR_AUC                         0.7290
# Val Loss                         0.9078
# -----------------------------------------
# Iter time: 182.8s
# ================================================================================
# ================================================================================
# [blur] Test   2/  5    Time elapsed: 1:01:51
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.5533
# AUROC                              0.6326
# PSNR_Acc                         0.4800
# PSNR_AUC                         0.6907
# Val Loss                         1.0546
# -----------------------------------------
# Iter time: 184.0s
# ================================================================================
# ================================================================================
# [blur] Test   3/  5    Time elapsed: 1:04:55
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.6200
# AUROC                              0.7026
# PSNR_Acc                         0.4867
# PSNR_AUC                         0.7434
# Val Loss                         0.9578
# -----------------------------------------
# Iter time: 178.4s
# ================================================================================
# ================================================================================
# [blur] Test   4/  5    Time elapsed: 1:07:54
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.5733
# AUROC                              0.6630
# PSNR_Acc                         0.4933
# PSNR_AUC                         0.7074
# Val Loss                         0.9574
# -----------------------------------------
# Iter time: 175.3s
# ================================================================================
# ================================================================================
# [blur] Test   5/  5    Time elapsed: 1:10:49
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.6267
# AUROC                              0.7276
# PSNR_Acc                         0.4800
# PSNR_AUC                         0.7659
# Val Loss                         0.8539
# -----------------------------------------
# Iter time: 178.6s
# ================================================================================

# --------------------------------------------------------------------------------
# AVERAGE (blur) over 5 runs
# --------------------------------------------------------------------------------
# Metric                        Avg Value
# ------------------------------------------
# Accuracy                         0.5920
# AUROC                              0.6879
# PSNR_Acc                         0.4827
# PSNR_AUC                         0.7273
# ------------------------------------------
# ################################################################################

# ################################################################################
# # ATTACK: random_crop
# ################################################################################
# ================================================================================
# [random_crop] Test   1/  5    Time elapsed: 1:13:47
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.8400
# AUROC                              0.9014
# PSNR_Acc                         0.5533
# PSNR_AUC                         0.9060
# Val Loss                         0.4226
# -----------------------------------------
# Iter time: 184.2s
# ================================================================================
# ================================================================================
# [random_crop] Test   2/  5    Time elapsed: 1:16:52
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.8667
# AUROC                              0.9296
# PSNR_Acc                         0.5667
# PSNR_AUC                         0.9076
# Val Loss                         0.3680
# -----------------------------------------
# Iter time: 185.1s
# ================================================================================
# ================================================================================
# [random_crop] Test   3/  5    Time elapsed: 1:19:57
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.8533
# AUROC                              0.8989
# PSNR_Acc                         0.5933
# PSNR_AUC                         0.9324
# Val Loss                         0.4236
# -----------------------------------------
# Iter time: 184.4s
# ================================================================================
# ================================================================================
# [random_crop] Test   4/  5    Time elapsed: 1:23:01
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.8400
# AUROC                              0.8968
# PSNR_Acc                         0.5867
# PSNR_AUC                         0.9490
# Val Loss                         0.4321
# -----------------------------------------
# Iter time: 184.2s
# ================================================================================
# ================================================================================
# [random_crop] Test   5/  5    Time elapsed: 1:26:05
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.8133
# AUROC                              0.8855
# PSNR_Acc                         0.6000
# PSNR_AUC                         0.9217
# Val Loss                         0.4641
# -----------------------------------------
# Iter time: 188.1s
# ================================================================================

# --------------------------------------------------------------------------------
# AVERAGE (random_crop) over 5 runs
# --------------------------------------------------------------------------------
# Metric                        Avg Value
# ------------------------------------------
# Accuracy                         0.8427
# AUROC                              0.9024
# PSNR_Acc                         0.5800
# PSNR_AUC                         0.9234
# ------------------------------------------
# ################################################################################

# ################################################################################
# # ATTACK: occlusion
# ################################################################################
# ================================================================================
# [occlusion] Test   1/  5    Time elapsed: 1:29:13
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.9467
# AUROC                              0.9809
# PSNR_Acc                         0.9133
# PSNR_AUC                         0.9957
# Val Loss                         0.1936
# -----------------------------------------
# Iter time: 178.2s
# ================================================================================
# ================================================================================
# [occlusion] Test   2/  5    Time elapsed: 1:32:12
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.9400
# AUROC                              0.9786
# PSNR_Acc                         0.9200
# PSNR_AUC                         0.9955
# Val Loss                         0.2032
# -----------------------------------------
# Iter time: 171.7s
# ================================================================================
# ================================================================================
# [occlusion] Test   3/  5    Time elapsed: 1:35:03
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.9267
# AUROC                              0.9761
# PSNR_Acc                         0.9333
# PSNR_AUC                         0.9925
# Val Loss                         0.2104
# -----------------------------------------
# Iter time: 171.6s
# ================================================================================
# ================================================================================
# [occlusion] Test   4/  5    Time elapsed: 1:37:55
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.9200
# AUROC                              0.9766
# PSNR_Acc                         0.9400
# PSNR_AUC                         0.9943
# Val Loss                         0.2093
# -----------------------------------------
# Iter time: 170.6s
# ================================================================================
# ================================================================================
# [occlusion] Test   5/  5    Time elapsed: 1:40:46
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.9200
# AUROC                              0.9791
# PSNR_Acc                         0.9200
# PSNR_AUC                         0.9914
# Val Loss                         0.2035
# -----------------------------------------
# Iter time: 173.4s
# ================================================================================

# --------------------------------------------------------------------------------
# AVERAGE (occlusion) over 5 runs
# --------------------------------------------------------------------------------
# Metric                        Avg Value
# ------------------------------------------
# Accuracy                         0.9307
# AUROC                              0.9783
# PSNR_Acc                         0.9253
# PSNR_AUC                         0.9939
# ------------------------------------------
# ################################################################################

# ################################################################################
# # ATTACK: geom_warp
# ################################################################################
# ================================================================================
# [geom_warp] Test   1/  5    Time elapsed: 1:43:39
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.8200
# AUROC                              0.8734
# PSNR_Acc                         0.5533
# PSNR_AUC                         0.9009
# Val Loss                         0.4691
# -----------------------------------------
# Iter time: 173.7s
# ================================================================================
# ================================================================================
# [geom_warp] Test   2/  5    Time elapsed: 1:46:33
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.7600
# AUROC                              0.8839
# PSNR_Acc                         0.5533
# PSNR_AUC                         0.8893
# Val Loss                         0.4278
# -----------------------------------------
# Iter time: 178.0s
# ================================================================================
# ================================================================================
# [geom_warp] Test   3/  5    Time elapsed: 1:49:31
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.8267
# AUROC                              0.8734
# PSNR_Acc                         0.5733
# PSNR_AUC                         0.8873
# Val Loss                         0.4509
# -----------------------------------------
# Iter time: 182.0s
# ================================================================================
# ================================================================================
# [geom_warp] Test   4/  5    Time elapsed: 1:52:33
# --------------------------------------------------------------------------------
                                                     
# Metric                            Value
# -----------------------------------------
# Accuracy                         0.8067
# AUROC                              0.8659
# PSNR_Acc                         0.5467
# PSNR_AUC                         0.9123
# Val Loss                         0.4644
# -----------------------------------------
# Iter time: 183.0s
# ================================================================================
# ================================================================================
# [geom_warp] Test   5/  5    Time elapsed: 1:55:36
# --------------------------------------------------------------------------------
#                                                      Metric                            Value
# -----------------------------------------
# Accuracy                         0.8333
# AUROC                              0.9014
# PSNR_Acc                         0.5600
# PSNR_AUC                         0.9342
# Val Loss                         0.4274
# -----------------------------------------
# Iter time: 185.2s
# ================================================================================

# --------------------------------------------------------------------------------
# AVERAGE (geom_warp) over 5 runs
# --------------------------------------------------------------------------------
# Metric                        Avg Value
# ------------------------------------------
# Accuracy                         0.8093
# AUROC                              0.8796
# PSNR_Acc                         0.5573
# PSNR_AUC                         0.9048
# ------------------------------------------
# ################################################################################

# ================================================================================
# All evaluations complete.
# ================================================================================